# 00 · F5 준비: 라벨 비율을 맞춘 오류 교정 재학습
F4는 운영 1.0이 원래 난이도 라벨과 다르게 예측한 질문만 학습했습니다. 운영 1.0은 원래보다 낮게 틀린 경우가 2.2배 많아서, 학습 데이터가 medium·hard 쪽으로 치우쳤습니다(easy 27%, 실제 49%). 그 결과 판단이 medium으로 쏠렸습니다.

F5는 F4의 오답을 **전부** 넣고, window마다 **맞게 예측한 질문을 라벨별로 더해 그 window의 실제 라벨 비율을 되돌립니다**. 복습도 F2·F4와 같은 크기(지난 cycle 96행 + 초기 학습 96행)이지만, 라벨별 균등이 아니라 실제 비율대로 뽑습니다. 그래서 학습 데이터 전체가 stream의 라벨 분포를 따릅니다. 로거 결과·stream·window·학습 설정은 F0~F4와 같습니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. Drive 연결 시 해당 계정을 선택하고 Google 화면에서 직접 승인합니다. GCP·Vertex 제출, GCS 업로드, 운영 `CURRENT` 변경 코드는 없습니다.

제공한 실행 패키지의 `retraining-effect-f5` 폴더를 `MyDrive/LLM_Router/` 아래에 업로드하세요. 봉인 test(`inputs/test.parquet`)는 이 폴더에 들어 있습니다. 첫 실험 폴더 `retraining-effect`의 로거 결과·run001, F3 폴더 `retraining-effect-f3`, F4 폴더 `retraining-effect-f4`는 읽기 전용으로 재사용합니다. 입력 데이터와 R0(운영 1.0) 모델은 **R1→R0 전이 실험 때 올린 `MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A`를 읽기 전용으로 재사용**합니다. 경로·SHA가 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f5")
FIRST = Path("/content/drive/MyDrive/LLM_Router/retraining-effect")  # 첫 실험: 로거·F1/F2 결과 재사용(읽기 전용)
F3_BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f3")  # F3 결과(있으면 비교에만 사용)
F4_BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f4")  # F4 결과(test·RouterArena 직접 비교)
SOURCE_ZIP = BASE / "source" / "retraining-source-f5.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "retraining-effect-v4-f5-run001"
RUN = BASE / "runs" / RUN_ID
LOG = FIRST / "logger" / "logger-log.parquet"
REFERENCES = [FIRST / "runs" / "retraining-effect-v1-run001", F3_BASE / "runs" / "retraining-effect-v2-f3-run001",
              F4_BASE / "runs" / "retraining-effect-v3-f4-run001"]
EXTRA = BASE / "inputs" / "extra-inputs.json"  # 봉인 test 7,202문항(이 kit에 포함)
# R1->R0 전이 실험 때 올린 입력과 복구된 anchor 모델을 읽기 전용으로 재사용한다.
INPUTS = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A")
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 SHA-256 파일이 필요합니다"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-retraining-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/retraining_effect"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config_f5.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
assert PINS.is_file(), f"전이 실험 입력이 필요합니다: {PINS}"
assert EXTRA.is_file(), f"kit의 봉인 test 입력이 필요합니다: {EXTRA}"
assert LOG.is_file() and (REFERENCES[0] / "plan.json").is_file(), "첫 실험 retraining-effect 폴더가 필요합니다"
print("Run directory:", RUN)


## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU **T4**를 선택합니다. 설치 후 이미 import한 패키지 버전이 다르면 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 실행하세요.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)

## anchor 확인과 계획 봉인
첫 실험과 같은 입력·로거·stream인지 확인한 뒤 F5 계획을 봉인합니다. 같은 RUN이 이미 있으면 검증 후 재사용합니다. window별 선택 수와 학습 데이터의 라벨 비율이 표로 나옵니다.

In [ ]:
import pandas as pd
subprocess.run([sys.executable, str(EXPERIMENT / "prepare_models.py"), "--inputs", str(INPUTS)], env=ENV, check=True)
if not RUN.exists():
    run("prepare", "--config", CONFIG, "--pins", PINS, "--data-dir", DATA, "--extra-inputs", EXTRA,
        "--logger-log", LOG, "--output-dir", RUN)
plan = json.loads((RUN / "plan.json").read_text())
reference = json.loads((REFERENCES[0] / "plan.json").read_text())
assert plan["logger_receipt"] == reference["logger_receipt"], "첫 실험과 로거가 다릅니다"
keys = ["sample_id", "window", "initial_label"]
ref_stream = pd.read_parquet(REFERENCES[0] / "shared-data/stream-windows.parquet", columns=keys)
new_stream = pd.read_parquet(RUN / "shared-data/stream-windows.parquet", columns=keys)
assert ref_stream.equals(new_stream), "첫 실험과 stream이 다릅니다"
print("stream 라벨 분포:", plan["stream_label_mix"])
selection = plan["matched_prior_selection"]
print("window별 선택 (errors = 원래 라벨과 다르게 예측, agreeing_fill = 비율을 맞추려 더한 맞게 예측한 질문)")
display(pd.DataFrame({w: {f"{l}/{k}": v[l][k] for l in v for k in ("errors", "agreeing_fill", "shortfall")}
                      for w, v in selection.items()}))
print("update별 학습 데이터 라벨 비율:")
mix = pd.DataFrame(plan["training_label_mix"]["f5_matched_prior"]).T
display((mix.T / mix.sum(1)).T.round(3).assign(rows=mix.sum(1)))
print("update별 학습량:")
display(pd.DataFrame(plan["training_budget"]["f5_matched_prior"]).T)
print("평가 역할 사이 텍스트 중복:", plan["pins"]["extra_inputs"]["evaluation_only_text_overlaps"])


## 다음 단계
정상 종료하면 `01_f5_matched_prior`를 GPU 세션에서 실행합니다.